# 안전괴리지수(SDI) 분석
## 제10차 산업안전보건실태조사 기반 산업안전 형식주의 측정

**분석 목표**: 형식적 안전관리(Formal)와 실질적 안전관리(Substantive)의 괴리를 수치화하여 업종별 비교 및 사고율과의 관계 검증

---

In [1]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                       'factor_analyzer', 'scikit-learn<1.6',
                       'statsmodels', 'matplotlib', 'seaborn', '-q'])
print('패키지 설치 완료')

패키지 설치 완료


## 1. 환경 설정 및 Import

In [2]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_kmo, calculate_bartlett_sphericity
import statsmodels.api as sm
from scipy import stats
import warnings, os

warnings.filterwarnings('ignore')

BASE = 'c:/Users/USER/Desktop/SAB_paper/data/raw/'
OUT  = 'c:/Users/USER/Desktop/SAB_paper/output/'
os.makedirs(OUT, exist_ok=True)

try:
    plt.rcParams['font.family'] = 'Malgun Gothic'
    plt.rcParams['axes.unicode_minus'] = False
except:
    pass

print(f'출력 디렉토리: {OUT}')

C:\Users\USER\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.4' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\USER\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


출력 디렉토리: c:/Users/USER/Desktop/SAB_paper/output/


## 2. 변수 정의 (PDF 확정 목록)

**F/S 변수 수 균형 처리**: 각 그룹별 composite 평균을 구해 F=7그룹, S=7그룹으로 균형화

In [3]:
# =================================================================
# SDI 변수 정의 — 선언(F) vs 운영(S) 이원 구조 (§ 제도적 분리 이론)
# F: 경영진 안전 선언·소통 채널 (Q17_1~6 / Q16_1~6) — 의례적 형식 신호
# S: 실제 운영 시스템·행동·효과 (Q17_7~17 / Q16_7~18) — 실질 작동 신호
# =================================================================

# 건설업 (Q16 기준)
F_CON_GROUPS = {
    'F-C_경영선언': ['Q16_1','Q16_2','Q16_3','Q16_4','Q16_5','Q16_6'],
}
S_CON_GROUPS = {
    'S-C_시스템보유': ['Q16_7','Q16_9','Q16_11','Q16_13'],
    'S-C_참여응답':  ['Q16_8'],
    'S-C_교육효과':  ['Q16_10'],
    'S-C_규정효과':  ['Q16_12'],
    'S-C_행동준수':  ['Q16_14','Q16_16','Q16_17','Q16_18'],
}

# 제조·서비스업 (Q17 기준)
F_MS_GROUPS = {
    'F-M_경영선언': ['Q17_1','Q17_2','Q17_3','Q17_4','Q17_5','Q17_6'],
}
S_MS_GROUPS = {
    'S-M_시스템보유': ['Q17_7','Q17_9','Q17_11','Q17_13'],
    'S-M_참여응답':  ['Q17_8'],
    'S-M_교육효과':  ['Q17_10'],
    'S-M_규정효과':  ['Q17_12'],
    'S-M_행동준수':  ['Q17_14','Q17_15','Q17_16','Q17_17'],
}

F_CON_ALL = [c for v in F_CON_GROUPS.values() for c in v]
S_CON_ALL = [c for v in S_CON_GROUPS.values() for c in v]
F_MS_ALL  = [c for v in F_MS_GROUPS.values() for c in v]
S_MS_ALL  = [c for v in S_MS_GROUPS.values() for c in v]

print(f'건설업   F(경영선언): {F_CON_ALL}')
print(f'건설업   S(운영실질): {S_CON_ALL}')
print(f'제조서비스 F(경영선언): {F_MS_ALL}')
print(f'제조서비스 S(운영실질): {S_MS_ALL}')
print(f'F 변수 수: {len(F_MS_ALL)}, S 변수 수: {len(S_MS_ALL)}')
print('-> SDI = S_score - F_score (운영실질 - 경영선언)',
      '| SDI<0: 선언>실질 = 형식주의 | SDI>0: 실질>선언 = 진짜 안전')

건설업   F(경영선언): ['Q16_1', 'Q16_2', 'Q16_3', 'Q16_4', 'Q16_5', 'Q16_6']
건설업   S(운영실질): ['Q16_7', 'Q16_9', 'Q16_11', 'Q16_13', 'Q16_8', 'Q16_10', 'Q16_12', 'Q16_14', 'Q16_16', 'Q16_17', 'Q16_18']
제조서비스 F(경영선언): ['Q17_1', 'Q17_2', 'Q17_3', 'Q17_4', 'Q17_5', 'Q17_6']
제조서비스 S(운영실질): ['Q17_7', 'Q17_9', 'Q17_11', 'Q17_13', 'Q17_8', 'Q17_10', 'Q17_12', 'Q17_14', 'Q17_15', 'Q17_16', 'Q17_17']
F 변수 수: 6, S 변수 수: 11
-> SDI = S_score - F_score (운영실질 - 경영선언) | SDI<0: 선언>실질 = 형식주의 | SDI>0: 실질>선언 = 진짜 안전


## 3. 데이터 로드 및 변수명 검증

In [4]:
df_con_raw = pd.read_csv(BASE + '제10차_산업안전보건_실태조사_raw_data_건설업_230824.CSV')
df_mfg_raw = pd.read_csv(BASE + '제10차_산업안전보건_실태조사_raw_data_제조업_230824.csv')
df_svc_raw = pd.read_csv(BASE + '제10차_산업안전보건_실태조사_raw_data_서비스업_230824.csv')
print(f'건설업: {df_con_raw.shape} | 제조업: {df_mfg_raw.shape} | 서비스업: {df_svc_raw.shape}')

def check_vars(df, cols, label):
    missing = [c for c in cols if c not in df.columns]
    if missing: print(f'  ⚠ [{label}] 없는 변수: {missing}')
    else:       print(f'  ✓ [{label}] 모두 존재')

print('\n변수 존재 확인:')
check_vars(df_con_raw, F_CON_ALL + S_CON_ALL, '건설업')
check_vars(df_mfg_raw, F_MS_ALL + S_MS_ALL,  '제조업')
check_vars(df_svc_raw, F_MS_ALL + S_MS_ALL,  '서비스업')

건설업: (1502, 181) | 제조업: (3255, 162) | 서비스업: (2551, 162)

변수 존재 확인:
  ✓ [건설업] 모두 존재
  ✓ [제조업] 모두 존재
  ✓ [서비스업] 모두 존재


## 4. 전처리: 무응답 처리 + 변수 방향 정렬 + 정규화

In [5]:
def minmax(s):
    # Min-Max 정규화 (NaN 유지)
    mn, mx = s.min(), s.max()
    if pd.isna(mn) or mx == mn:
        return pd.Series(np.where(s.isna(), np.nan, 0.0), index=s.index)
    return (s - mn) / (mx - mn)

def preprocess_con(df):
    d = df.copy()
    # Q16 전 문항 NaN 처리 (F+S 모두)
    q16_all = [f'Q16_{i}' for i in range(1, 19)]
    for col in q16_all:
        if col in d.columns: d[col] = d[col].replace(9, np.nan)
    # 이진 통제변수 NaN 처리
    for col in ['Q6','Q9','Q12_1','Q12_2','Q10','Q18_1']:
        if col in d.columns: d[col] = d[col].replace(9, np.nan)
    # 인원수·사고건수·안전관리비
    for col in ['Q8_1','Q8_2','Q8_3','Q8_4','Q4_1','Q4_2','Q4_3','Q4_4','Q25']:
        if col in d.columns: d[col] = d[col].replace([999, 99999], np.nan)
    q27 = [c for c in d.columns if c.startswith('Q27_')]
    d[q27] = d[q27].replace([999, 99999], np.nan)
    # 이진 방향 정렬 (통제변수)
    for col in ['Q6','Q9','Q12_1','Q12_2']:
        if col in d.columns: d[col] = d[col].map({1:1.0, 2:0.0})
    d['Q10'] = d['Q10'].map({1:1.0, 2:0.5, 3:0.0, 4:np.nan})
    d['Q18_1'] = d['Q18_1'].map({1:1.0, 2:0.0})
    # SDI 변수 정규화 (F+S 모두 Min-Max)
    for col in F_CON_ALL + S_CON_ALL:
        if col in d.columns: d[col] = minmax(d[col])
    return d

def preprocess_ms(df):
    d = df.copy()
    # Q17 전 문항 NaN 처리
    q17_all = [f'Q17_{i}' for i in range(1, 18)]
    for col in q17_all:
        if col in d.columns: d[col] = d[col].replace(9, np.nan)
    # 이진 통제변수 NaN 처리
    for col in ['Q6','Q9_1','Q9_2','Q11','Q13_1','Q13_2','Q10','Q19_1','Q22_1','Q20_1']:
        if col in d.columns: d[col] = d[col].replace(9, np.nan)
    # 인원수·사고건수
    for col in ['Q8_1','Q8_2','Q1_1','Q1_2','Q1_3','Q1_4','Q3_1_1']:
        if col in d.columns: d[col] = d[col].replace([999, 99999], np.nan)
    q27 = [c for c in d.columns if c.startswith('Q27_')]
    d[q27] = d[q27].replace([999, 99999], np.nan)
    # 이진 방향 정렬
    for col in ['Q6','Q11','Q13_1','Q13_2','Q9_1','Q9_2']:
        if col in d.columns: d[col] = d[col].map({1:1.0, 2:0.0})
    d['Q10'] = d['Q10'].map({1:1.0, 2:0.0, 3:np.nan})
    # SDI 변수 정규화
    for col in F_MS_ALL + S_MS_ALL:
        if col in d.columns: d[col] = minmax(d[col])
    return d

df_con = preprocess_con(df_con_raw)
df_mfg = preprocess_ms(df_mfg_raw)
df_svc = preprocess_ms(df_svc_raw)
print('전처리 완료')
print(f'  건설업 결측률 - F: {df_con[F_CON_ALL].isna().mean().mean():.2%}, S: {df_con[S_CON_ALL].isna().mean().mean():.2%}')
print(f'  제조업 결측률 - F: {df_mfg[F_MS_ALL].isna().mean().mean():.2%}, S: {df_mfg[S_MS_ALL].isna().mean().mean():.2%}')

전처리 완료
  건설업 결측률 - F: 0.00%, S: 0.00%
  제조업 결측률 - F: 0.00%, S: 0.00%


## 5. SDI 산출

**계산 방식**: 각 그룹 내 변수들의 행 평균 → 그룹 composite (0~1) → 7개 F composite 평균 = F_score, 7개 S composite 평균 = S_score → **SDI = S_score − F_score**

In [6]:
def compute_sdi(df, f_groups, s_groups, wt_col='WT2', label=''):
    """그룹 composite 기반 SDI 산출"""
    f_comp, s_comp = {}, {}
    for name, cols in f_groups.items():
        exist = [c for c in cols if c in df.columns]
        f_comp[name] = df[exist].mean(axis=1)  # 행 평균 (이미 0~1)
    for name, cols in s_groups.items():
        exist = [c for c in cols if c in df.columns]
        s_comp[name] = df[exist].mean(axis=1)

    f_df = pd.DataFrame(f_comp)
    s_df = pd.DataFrame(s_comp)
    F  = f_df.mean(axis=1)   # 7개 그룹 평균
    S  = s_df.mean(axis=1)
    sdi = S - F               # SDI = Substantive - Formal

    result = pd.DataFrame({'F_score': F, 'S_score': S, 'SDI': sdi})
    result = pd.concat([result, f_df, s_df], axis=1)

    print(f'\n=== {label} SDI ===')
    print(f'  F score: mean={F.mean():.4f}  std={F.std():.4f}')
    print(f'  S score: mean={S.mean():.4f}  std={S.std():.4f}')
    print(f'  SDI    : mean={sdi.mean():.4f}  median={sdi.median():.4f}  std={sdi.std():.4f}')
    neg = (sdi < 0).sum(); pos = (sdi > 0).sum(); tot = sdi.notna().sum()
    print(f'  SDI<0 (형식주의): {neg}개 ({neg/tot*100:.1f}%)')
    print(f'  SDI>0 (실질우세): {pos}개 ({pos/tot*100:.1f}%)')
    if wt_col in df.columns:
        mask = sdi.notna() & df[wt_col].notna()
        wm = np.average(sdi[mask], weights=df.loc[mask, wt_col])
        print(f'  가중 SDI 평균(WT2): {wm:.4f}')
    return result

sdi_con = compute_sdi(df_con, F_CON_GROUPS, S_CON_GROUPS, label='건설업')
sdi_mfg = compute_sdi(df_mfg, F_MS_GROUPS, S_MS_GROUPS, label='제조업')
sdi_svc = compute_sdi(df_svc, F_MS_GROUPS, S_MS_GROUPS, label='서비스업')

for df_, sdi_ in [(df_con, sdi_con), (df_mfg, sdi_mfg), (df_svc, sdi_svc)]:
    df_['F_score'] = sdi_['F_score']
    df_['S_score'] = sdi_['S_score']
    df_['SDI']     = sdi_['SDI']


=== 건설업 SDI ===
  F score: mean=0.8767  std=0.1463
  S score: mean=0.8275  std=0.1551
  SDI    : mean=-0.0492  median=-0.0333  std=0.0921
  SDI<0 (형식주의): 949개 (63.2%)
  SDI>0 (실질우세): 273개 (18.2%)
  가중 SDI 평균(WT2): -0.0510

=== 제조업 SDI ===
  F score: mean=0.8479  std=0.1440
  S score: mean=0.8076  std=0.1508
  SDI    : mean=-0.0403  median=-0.0208  std=0.0966
  SDI<0 (형식주의): 1936개 (59.5%)
  SDI>0 (실질우세): 842개 (25.9%)
  가중 SDI 평균(WT2): -0.0406

=== 서비스업 SDI ===
  F score: mean=0.8556  std=0.1473
  S score: mean=0.8054  std=0.1648
  SDI    : mean=-0.0502  median=-0.0250  std=0.0966
  SDI<0 (형식주의): 1535개 (60.2%)
  SDI>0 (실질우세): 508개 (19.9%)
  가중 SDI 평균(WT2): -0.0510


## 6. 탐색적 요인분석 (EFA) — 구성타당도 검증

**대상**: Substantive Likert 변수 (Binary/Count Formal 제외, 척도 동질성 확보)  
**방법**: Principal Axis Factoring + Varimax 회전 (2요인 가정)

In [7]:
def run_efa(df, items, label, n_factors=2):
    items_exist = [c for c in items if c in df.columns]
    X = df[items_exist].dropna()
    print(f'\n=== EFA: {label} === (유효 n={len(X)}, 변수 수={len(items_exist)})')

    # KMO & Bartlett
    _, kmo = calculate_kmo(X)
    chi2, p = calculate_bartlett_sphericity(X)
    kmo_grade = '우수(≥0.9)' if kmo>=0.9 else '양호(0.8~)' if kmo>=0.8 else '보통(0.7~)' if kmo>=0.7 else '미흡'
    print(f'  KMO: {kmo:.3f} [{kmo_grade}]')
    print(f'  Bartlett: p={"<0.001" if p<0.001 else f"{p:.4f}"} [{"EFA 적합" if p<0.05 else "EFA 부적합"}]')

    # 요인 수 탐색 (고유값 > 1)
    fa_full = FactorAnalyzer(n_factors=min(len(items_exist)-1, 20), method='principal', rotation=None)
    fa_full.fit(X)
    ev = fa_full.get_eigenvalues()[0]
    n_kaiser = int(sum(ev > 1))
    var_2f = sum(ev[:2]) / sum(ev) * 100
    print(f'  고유값>1 요인 수: {n_kaiser}개  |  처음 5개: {np.round(ev[:5], 2)}')
    print(f'  2요인 누적 분산 설명: {var_2f:.1f}%')

    # 2요인 EFA (Varimax)
    fa2 = FactorAnalyzer(n_factors=n_factors, method='principal', rotation='varimax')
    fa2.fit(X)
    loads = pd.DataFrame(fa2.loadings_, index=items_exist, columns=['Factor1', 'Factor2'])
    loads['주요요인'] = loads.abs().idxmax(axis=1)
    loads['최대적재'] = loads[['Factor1','Factor2']].abs().max(axis=1)

    strong = loads[loads['최대적재'] >= 0.4]
    print(f'\n  [적재량 ≥ 0.4 변수 {len(strong)}개]')
    print(strong[['Factor1','Factor2','주요요인','최대적재']].round(3).to_string())
    return loads

loads_con = run_efa(df_con, S_CON_ALL, '건설업 Substantive')
loads_ms  = run_efa(df_mfg, S_MS_ALL,  '제조업 Substantive')


=== EFA: 건설업 Substantive === (유효 n=1502, 변수 수=11)
  KMO: 0.950 [우수(≥0.9)]
  Bartlett: p=<0.001 [EFA 적합]
  고유값>1 요인 수: 1개  |  처음 5개: [6.86 0.78 0.63 0.47 0.43]
  2요인 누적 분산 설명: 69.5%

  [적재량 ≥ 0.4 변수 11개]


        Factor1  Factor2     주요요인   최대적재
Q16_7     0.771    0.269  Factor1  0.771
Q16_9     0.789    0.324  Factor1  0.789
Q16_11    0.779    0.351  Factor1  0.779
Q16_13    0.620    0.465  Factor1  0.620
Q16_8     0.774    0.316  Factor1  0.774
Q16_10    0.743    0.400  Factor1  0.743
Q16_12    0.734    0.410  Factor1  0.734
Q16_14    0.343    0.780  Factor2  0.780
Q16_16    0.406    0.788  Factor2  0.788
Q16_17    0.501    0.508  Factor2  0.508
Q16_18    0.302    0.832  Factor2  0.832

=== EFA: 제조업 Substantive === (유효 n=3255, 변수 수=11)
  KMO: 0.944 [우수(≥0.9)]
  Bartlett: p=<0.001 [EFA 적합]
  고유값>1 요인 수: 1개  |  처음 5개: [6.45 0.77 0.64 0.54 0.53]
  2요인 누적 분산 설명: 65.6%

  [적재량 ≥ 0.4 변수 11개]
        Factor1  Factor2     주요요인   최대적재
Q17_7     0.756    0.221  Factor1  0.756
Q17_9     0.761    0.281  Factor1  0.761
Q17_11    0.683    0.438  Factor1  0.683
Q17_13    0.577    0.474  Factor1  0.577
Q17_8     0.721    0.325  Factor1  0.721
Q17_10    0.691    0.406  Factor1  0.691
Q17_12    0.677  

## 7. 업종 비교 및 시각화

In [8]:
df_con['industry'] = '건설'
df_mfg['industry'] = '제조'
df_svc['industry'] = '서비스'

sdi_all = pd.concat([
    df_con[['SDI','F_score','S_score','WT2','industry']],
    df_mfg[['SDI','F_score','S_score','WT2','industry']],
    df_svc[['SDI','F_score','S_score','WT2','industry']],
], ignore_index=True)

# 기술통계
print('=== 업종별 SDI 기술통계 (비가중) ===')
desc = sdi_all.groupby('industry')['SDI'].agg(
    N='count', 평균='mean', 중앙값='median', 표준편차='std', 최솟값='min', 최댓값='max'
).round(4)
print(desc.to_string())

# 가중 평균
print('\n가중 SDI 평균 (WT2):')
for ind in ['건설','제조','서비스']:
    sub = sdi_all[sdi_all.industry == ind]
    mask = sub['SDI'].notna() & sub['WT2'].notna()
    wm = np.average(sub.loc[mask,'SDI'], weights=sub.loc[mask,'WT2'])
    print(f'  {ind}: {wm:.4f}')

# Kruskal-Wallis 검정
g = [sdi_all.loc[sdi_all.industry==k,'SDI'].dropna() for k in ['건설','제조','서비스']]
stat, pval = stats.kruskal(*g)
print(f'\nKruskal-Wallis: H={stat:.3f}, p={"<0.001" if pval<0.001 else f"{pval:.4f}"}')
print('→ 업종 간 SDI 분포 차이' + (' 유의함 (p<0.05)' if pval<0.05 else ' 유의하지 않음'))

# 사후 검정 (Mann-Whitney)
print('\nMann-Whitney 쌍별 비교:')
from itertools import combinations
pairs = list(combinations(['건설','제조','서비스'], 2))
for a, b in pairs:
    ga = sdi_all.loc[sdi_all.industry==a,'SDI'].dropna()
    gb = sdi_all.loc[sdi_all.industry==b,'SDI'].dropna()
    u, p2 = stats.mannwhitneyu(ga, gb, alternative='two-sided')
    sig = '***' if p2<0.001 else '**' if p2<0.01 else '*' if p2<0.05 else 'n.s.'
    print(f'  {a} vs {b}: p={p2:.4f} {sig}')

# 박스플롯
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SDI 박스플롯
order = ['건설','제조','서비스']
colors = ['#E74C3C','#3498DB','#2ECC71']
data_by_ind = [sdi_all.loc[sdi_all.industry==k,'SDI'].dropna().values for k in order]
bp = axes[0].boxplot(data_by_ind, labels=order, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], colors):
    patch.set_facecolor(color); patch.set_alpha(0.7)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5, label='SDI=0 (균형점)')
axes[0].set_title('업종별 SDI 분포', fontsize=13, fontweight='bold')
axes[0].set_xlabel('업종'); axes[0].set_ylabel('SDI (Substantive - Formal)')
axes[0].legend()

# F/S 점수 비교 (막대)
means = sdi_all.groupby('industry')[['F_score','S_score']].mean().loc[order]
x = np.arange(len(order)); w = 0.35
axes[1].bar(x-w/2, means['F_score'], w, label='Formal(형식)', color='#E74C3C', alpha=0.8)
axes[1].bar(x+w/2, means['S_score'], w, label='Substantive(실질)', color='#2ECC71', alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(order)
axes[1].set_title('Formal vs Substantive 평균 점수', fontsize=13, fontweight='bold')
axes[1].set_ylabel('평균 점수 (0~1)'); axes[1].legend()

plt.tight_layout()
plt.savefig(OUT+'sdi_comparison.png', dpi=150, bbox_inches='tight')
plt.close()
print('\n그래프 저장: output/sdi_comparison.png')

=== 업종별 SDI 기술통계 (비가중) ===
             N      평균     중앙값    표준편차     최솟값     최댓값
industry                                              
건설        1502 -0.0492 -0.0333  0.0921 -0.5042  0.3458
서비스       2551 -0.0502 -0.0250  0.0966 -0.5292  0.3333
제조        3255 -0.0403 -0.0208  0.0966 -0.5000  0.3583

가중 SDI 평균 (WT2):
  건설: -0.0510
  제조: -0.0406
  서비스: -0.0510

Kruskal-Wallis: H=19.825, p=<0.001
→ 업종 간 SDI 분포 차이 유의함 (p<0.05)

Mann-Whitney 쌍별 비교:
  건설 vs 제조: p=0.0000 ***
  건설 vs 서비스: p=0.1640 n.s.
  제조 vs 서비스: p=0.0017 **



그래프 저장: output/sdi_comparison.png


## 8. Zero-Inflated Poisson(ZIP) 회귀 — 건설업

**모형 선택 근거**: 사고 0건 비율 64.8% → 두 종류의 0이 혼재
- **구조적 0 (Structural zero)**: 위험 노출 자체가 없어 사고가 날 수 없는 사업장
- **표본 0 (Sampling zero)**: 위험은 있으나 조사 기간 중 우연히 0건

**방정식 구성**:
- **Count 방정식**: SDI + 통제변수 → 사고 발생률(λ)에 미치는 영향 (주요 관심)
- **Inflation 방정식**: `log_workers` → 구조적 무사고 집단 확률(π) 결정 (소규모 사업장일수록 π↑)

**offset**: log(Q4_1) — 주간 평균 종사자 수 (사고 노출량 통제)

In [9]:
import statsmodels.api as sm

ACC_COLS = ['Q27_1_1','Q27_1_2','Q27_1_3','Q27_3_1','Q27_3_2','Q27_3_3']

def make_accident_total(df):
    sub = df[[c for c in ACC_COLS if c in df.columns]].copy()
    all_nan = sub.isna().all(axis=1)
    total = sub.fillna(0).sum(axis=1).astype(float)
    total[all_nan] = np.nan
    return total

def run_glm_nb(df, y_col, count_vars, offset_col, label):
    """
    음이항 GLM 회귀 (GLM with NegativeBinomial family, log link, offset=log_workers)
    ZIP 대비 장점: 구조적 영점(structural zero) 가정 불필요, 수렴 안정
    """
    avail = [c for c in count_vars if c in df.columns]
    needed = list(dict.fromkeys([y_col, offset_col] + avail))
    data = df[[c for c in needed if c in df.columns]].replace([np.inf, -np.inf], np.nan).dropna()
    print(f'\n=== {label} GLM-NegBin 회귀 === (유효 n={len(data)})')
    zero_pct = (data[y_col] == 0).mean() * 100
    print(f'  사고 0건 비율: {zero_pct:.1f}%')
    print(f'  독립변수 ({len(avail)}개): {avail}')
    if len(data) == 0:
        print('  유효 데이터 없음'); return None, None

    y = data[y_col].astype(float)
    X = sm.add_constant(data[avail].astype(float))
    offset = data[offset_col].astype(float)

    try:
        model = sm.GLM(y, X, family=sm.families.NegativeBinomial(), offset=offset)
        res = model.fit()

        params = res.params.drop('const', errors='ignore')
        pvals  = res.pvalues.drop('const', errors='ignore')
        irr    = np.exp(params)
        ci     = np.exp(res.conf_int().drop('const', errors='ignore'))
        const_b = res.params.get('const', np.nan)

        rows = [{'변수': 'const', '계수(β)': round(const_b, 4), 'IRR': round(np.exp(const_b), 4),
                 'CI_95%_lower': np.nan, 'CI_95%_upper': np.nan,
                 'p값': round(res.pvalues.get('const', np.nan), 4), '유의성': ''}]
        for nm in params.index:
            p = float(pvals.get(nm, np.nan))
            rows.append({
                '변수': nm, '계수(β)': round(float(params[nm]), 4), 'IRR': round(float(irr[nm]), 4),
                'CI_95%_lower': round(float(ci.loc[nm, 0]), 4), 'CI_95%_upper': round(float(ci.loc[nm, 1]), 4),
                'p값': round(p, 4),
                '유의성': '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else ''
            })
        summ = pd.DataFrame(rows)

        sdi_row = summ[summ['변수'] == 'SDI']
        if not sdi_row.empty:
            r = sdi_row.iloc[0]
            print(f'\n  [SDI 핵심 결과]')
            print(f'    β={r["계수(β)"]:.4f}, IRR={r["IRR"]:.4f}, '
                  f'95%CI=[{r["CI_95%_lower"]:.4f}, {r["CI_95%_upper"]:.4f}], p={r["p값"]:.4f} {r["유의성"]}')
            direction = '↑ 사고감소 (가설 지지 ✓)' if r['계수(β)'] < 0 else '↑ 사고증가 (가설 불일치 ✗)'
            print(f'    SDI↑ → {direction}')

        return summ, res
    except Exception as e:
        print(f'  GLM-NegBin 오류: {e}'); return None, None

In [10]:
# ---- 건설업 회귀 데이터 준비 ----
reg_con = df_con.copy()
reg_con['total_accident'] = make_accident_total(reg_con)
reg_con['log_workers']    = np.log(reg_con['Q4_1'].clip(1))

# 제도적 형식주의 통제변수 (SDI와 독립적인 인증·위원회 이진 변수)
reg_con['formal_inst'] = reg_con[['Q6','Q9','Q10','Q12_1','Q12_2']].mean(axis=1)
reg_con['edu_impl']    = reg_con['Q18_1']  # 교육실시 여부
reg_con['log_safecost'] = np.log(reg_con['Q25'].replace([999, 99999, 0], np.nan).clip(1))

# 근로자 구성 비율
reg_con['ratio_aged']    = (reg_con['Q4_2'] / reg_con['Q4_1'].clip(1)).clip(0, 1)
reg_con['ratio_foreign'] = (reg_con['Q4_3'] / reg_con['Q4_1'].clip(1)).clip(0, 1)
reg_con['ratio_female']  = (reg_con['Q4_4'] / reg_con['Q4_1'].clip(1)).clip(0, 1)

# 범주형 더미화
for cat_col, new_prefix in [('Q14','Q14d'),('Q22','Q22d')]:
    if cat_col in reg_con.columns:
        dummies = pd.get_dummies(reg_con[cat_col], prefix=new_prefix, drop_first=True, dtype=float)
        reg_con = pd.concat([reg_con, dummies], axis=1)

for cat_col, new_prefix in [('SQ2','SQ2d'),('Q2','Q2d')]:
    if cat_col in reg_con.columns:
        top_cats = reg_con[cat_col].value_counts().head(4).index
        temp = reg_con[cat_col].where(reg_con[cat_col].isin(top_cats), other=99)
        dummies = pd.get_dummies(temp, prefix=new_prefix, drop_first=True, dtype=float)
        reg_con = pd.concat([reg_con, dummies], axis=1)

count_vars_con = ['SDI', 'formal_inst', 'edu_impl', 'log_safecost',
                   'ratio_aged', 'ratio_foreign', 'ratio_female']
count_vars_con += [c for c in reg_con.columns if c.startswith('Q14d_')][:3]
count_vars_con += [c for c in reg_con.columns if c.startswith('Q22d_')][:3]
count_vars_con += [c for c in reg_con.columns if c.startswith('SQ2d_')][:3]
count_vars_con += [c for c in reg_con.columns if c.startswith('Q2d_')][:4]

summ_con, res_con = run_glm_nb(
    reg_con, 'total_accident', count_vars_con, 'log_workers', '건설업')


=== 건설업 GLM-NegBin 회귀 === (유효 n=1482)
  사고 0건 비율: 64.8%
  독립변수 (19개): ['SDI', 'formal_inst', 'edu_impl', 'log_safecost', 'ratio_aged', 'ratio_foreign', 'ratio_female', 'Q14d_2.0', 'Q14d_3.0', 'Q22d_2', 'Q22d_3', 'Q22d_4', 'SQ2d_2', 'SQ2d_3', 'SQ2d_4', 'Q2d_3', 'Q2d_5', 'Q2d_10', 'Q2d_99']

  [SDI 핵심 결과]
    β=0.8381, IRR=2.3120, 95%CI=[0.9822, 5.4422], p=0.0550 
    SDI↑ → ↑ 사고증가 (가설 불일치 ✗)


## 9. Zero-Inflated Poisson(ZIP) 회귀 — 제조·서비스업

**모형 선택 근거**: 사고 0건 비율 80.4% → 구조적 0 혼재 가능성이 건설업보다 높음

**방정식 구성**:
- **Count 방정식**: SDI + 통제변수 → 사고 발생률(λ)
- **Inflation 방정식**: `log_workers` + `union` → 구조적 무사고 확률(π)
  - 소규모 사업장(log_workers↓)은 위험 노출이 적어 구조적 무사고 가능성 ↑
  - 노조 유무(union)는 안전관리 체계성과 연관 — 구조적 0 확률에 영향

**offset**: log(Q1_1) — 전체 종사자 수

In [11]:
# ---- 제조·서비스업 합산 ----
reg_ms = pd.concat([df_mfg.copy(), df_svc.copy()], ignore_index=True)
reg_ms['total_accident'] = make_accident_total(reg_ms)
reg_ms['log_workers']    = np.log(reg_ms['Q1_1'].clip(1))

# 제도적 형식주의 통제변수
reg_ms['formal_inst'] = reg_ms[['Q6','Q9_1','Q9_2','Q10','Q11','Q13_1','Q13_2']].mean(axis=1)
reg_ms['edu_impl']     = reg_ms['Q19_1'].map({1:1.0, 2:0.0}) if 'Q19_1' in reg_ms.columns else np.nan
reg_ms['workenv_meas'] = reg_ms['Q22_1'].map({1:1.0, 2:0.0, 3:0.0, 4:np.nan}) if 'Q22_1' in reg_ms.columns else np.nan
reg_ms['health_chk']   = reg_ms['Q20_1'].map({1:1.0, 2:0.0}) if 'Q20_1' in reg_ms.columns else np.nan

# 근로자 구성 비율
for col_n, col_d, newcol in [('Q1_2','Q1_1','ratio_aged'),
                               ('Q1_3','Q1_1','ratio_foreign'),
                               ('Q1_4','Q1_1','ratio_female'),
                               ('Q3_1_1','Q1_1','ratio_night')]:
    if col_n in reg_ms.columns:
        reg_ms[newcol] = (reg_ms[col_n] / reg_ms['Q1_1'].clip(1)).clip(0, 1)

# 더미화
for cat_col, new_prefix in [('Q15','Q15d'),('SQ4','SQ4d'),('Q2','Q2dms')]:
    if cat_col in reg_ms.columns:
        dummies = pd.get_dummies(reg_ms[cat_col], prefix=new_prefix, drop_first=True, dtype=float)
        reg_ms = pd.concat([reg_ms, dummies], axis=1)

count_vars_ms = ['SDI', 'formal_inst', 'edu_impl', 'workenv_meas', 'health_chk',
                  'ratio_aged', 'ratio_foreign', 'ratio_female', 'ratio_night']
count_vars_ms += [c for c in reg_ms.columns if c.startswith('Q15d_')][:4]
count_vars_ms += [c for c in reg_ms.columns if c.startswith('SQ4d_')][:2]
count_vars_ms += [c for c in reg_ms.columns if c.startswith('Q2dms_')][:3]

summ_ms, res_ms = run_glm_nb(
    reg_ms, 'total_accident', count_vars_ms, 'log_workers', '제조·서비스업')


=== 제조·서비스업 GLM-NegBin 회귀 === (유효 n=5327)
  사고 0건 비율: 80.4%
  독립변수 (13개): ['SDI', 'formal_inst', 'edu_impl', 'workenv_meas', 'health_chk', 'ratio_aged', 'ratio_foreign', 'ratio_female', 'ratio_night', 'Q15d_2.0', 'Q15d_3.0', 'SQ4d_2', 'Q2dms_2']

  [SDI 핵심 결과]
    β=-1.8174, IRR=0.1625, 95%CI=[0.0935, 0.2823], p=0.0000 ***
    SDI↑ → ↑ 사고감소 (가설 지지 ✓)


## 10. CSV 출력

In [12]:
# 1. 변수 분류표 - 건설업
rows = []
for name, cols in {**F_CON_GROUPS, **S_CON_GROUPS}.items():
    cat = 'Formal' if name.startswith('F') else 'Substantive'
    rows.append({'그룹ID': name, '원본변수코드': ', '.join(cols), '범주': cat, '변수수': len(cols)})
pd.DataFrame(rows).to_csv(OUT+'variable_classification_con.csv', index=False, encoding='utf-8-sig')

# 2. 변수 분류표 - 제조·서비스업
rows = []
for name, cols in {**F_MS_GROUPS, **S_MS_GROUPS}.items():
    cat = 'Formal' if name.startswith('F') else 'Substantive'
    rows.append({'그룹ID': name, '원본변수코드': ', '.join(cols), '범주': cat, '변수수': len(cols)})
pd.DataFrame(rows).to_csv(OUT+'variable_classification_ms.csv', index=False, encoding='utf-8-sig')

# 3-4. EFA 적재량
if loads_con is not None:
    loads_con.reset_index().rename(columns={'index':'원본변수코드'}).to_csv(
        OUT+'efa_loadings_con.csv', index=False, encoding='utf-8-sig')
if loads_ms is not None:
    loads_ms.reset_index().rename(columns={'index':'원본변수코드'}).to_csv(
        OUT+'efa_loadings_ms.csv', index=False, encoding='utf-8-sig')

# 5-7. SDI 점수 (ID + 핵심 지수)
for df_, sdi_, fname in [(df_con, sdi_con, 'sdi_scores_con'),
                          (df_mfg, sdi_mfg, 'sdi_scores_mfg'),
                          (df_svc, sdi_svc, 'sdi_scores_svc')]:
    out_sdi = pd.DataFrame({'row_id': range(len(sdi_))}).set_index('row_id')
    out_sdi['F_score'] = sdi_['F_score'].values
    out_sdi['S_score'] = sdi_['S_score'].values
    out_sdi['SDI']     = sdi_['SDI'].values
    if 'WT2' in df_.columns:
        out_sdi['WT2'] = df_['WT2'].values
    out_sdi.to_csv(OUT+fname+'.csv', encoding='utf-8-sig')

# 8. 업종별 기술통계
desc = sdi_all.groupby('industry')['SDI'].agg(
    N='count', 평균='mean', 중앙값='median', 표준편차='std', 최솟값='min', 최댓값='max'
).round(4)
wm_dict = {}
for ind in ['건설','제조','서비스']:
    sub = sdi_all[sdi_all.industry==ind]
    mask = sub['SDI'].notna() & sub['WT2'].notna()
    wm_dict[ind] = round(np.average(sub.loc[mask,'SDI'], weights=sub.loc[mask,'WT2']), 4)
desc['가중평균(WT2)'] = pd.Series(wm_dict)
desc.to_csv(OUT+'sdi_descriptive_stats.csv', encoding='utf-8-sig')

# 9-10. 회귀분석 결과
if summ_con is not None:
    summ_con.to_csv(OUT+'regression_results_con.csv', index=False, encoding='utf-8-sig')
if summ_ms is not None:
    summ_ms.to_csv(OUT+'regression_results_ms.csv', index=False, encoding='utf-8-sig')

print('=== CSV 출력 완료 ===')
for f in sorted(os.listdir(OUT)):
    if f.endswith('.csv'):
        fpath = OUT + f
        size = os.path.getsize(fpath)
        print(f'  {f} ({size:,} bytes)')

=== CSV 출력 완료 ===


  efa_loadings_con.csv (866 bytes)
  efa_loadings_ms.csv (863 bytes)
  regression_results_con.csv (976 bytes)
  regression_results_ms.csv (706 bytes)
  sdi_descriptive_stats.csv (257 bytes)
  sdi_scores_con.csv (75,764 bytes)
  sdi_scores_mfg.csv (172,938 bytes)
  sdi_scores_svc.csv (129,896 bytes)
  variable_classification_con.csv (366 bytes)
  variable_classification_ms.csv (366 bytes)


## 11. 마크다운 보고서 자동 생성

In [13]:
# ---- 보고서 데이터 수집 ----
kmo_con_val = None; kmo_ms_val = None
ev_con_top = None; ev_ms_top = None
try:
    _, kmo_con_val = calculate_kmo(df_con[S_CON_ALL].dropna())
    fa_tmp = FactorAnalyzer(n_factors=min(len(S_CON_ALL)-1,20), method='principal', rotation=None)
    fa_tmp.fit(df_con[S_CON_ALL].dropna())
    ev_con_top = np.round(fa_tmp.get_eigenvalues()[0][:5], 2).tolist()
except: pass
try:
    _, kmo_ms_val = calculate_kmo(df_mfg[S_MS_ALL].dropna())
    fa_tmp = FactorAnalyzer(n_factors=min(len(S_MS_ALL)-1,20), method='principal', rotation=None)
    fa_tmp.fit(df_mfg[S_MS_ALL].dropna())
    ev_ms_top = np.round(fa_tmp.get_eigenvalues()[0][:5], 2).tolist()
except: pass

desc_stats = sdi_all.groupby('industry')['SDI'].agg(
    N='count', 평균='mean', 중앙값='median', 표준편차='std'
).round(4)
wm_vals = {}
for ind in ['건설','제조','서비스']:
    sub = sdi_all[sdi_all.industry==ind]
    mask = sub['SDI'].notna() & sub['WT2'].notna()
    wm_vals[ind] = round(np.average(sub.loc[mask,'SDI'], weights=sub.loc[mask,'WT2']), 4)

g = [sdi_all.loc[sdi_all.industry==k,'SDI'].dropna() for k in ['건설','제조','서비스']]
kw_stat, kw_p = stats.kruskal(*g)

sdi_con_dir = '↓ 음수 (형식 우세)' if df_con['SDI'].mean() < 0 else '↑ 양수 (실질 우세)'
sdi_mfg_dir = '↓ 음수 (형식 우세)' if df_mfg['SDI'].mean() < 0 else '↑ 양수 (실질 우세)'
sdi_svc_dir = '↓ 음수 (형식 우세)' if df_svc['SDI'].mean() < 0 else '↑ 양수 (실질 우세)'

# ---- 보고서 빌드 ----
lines = []
lines.append('# 안전괴리지수(SDI) 분석 보고서\n')
lines.append('> **제10차 산업안전보건실태조사** 기반 산업안전 형식주의 측정 분석  \n')
lines.append(f'> 분석일: 2026-05-22  \n')
lines.append(f'> 데이터: 건설업 n=1,502 / 제조업 n=3,255 / 서비스업 n=2,551 (총 7,308개 사업장)\n\n')
lines.append('---\n\n')

# 1. 연구 목적
lines.append('## 1. 연구 목적 및 SDI 개념\n\n')
lines.append('본 연구는 기업이 규제 대응을 위해 **서류상 안전(형식적 안전관리, Formal)**에는 투자하면서도 **실제 사고를 줄이는 안전(실질적 안전관리, Substantive)**은 방치하는 *안전 형식주의(Safety Decoupling)* 현상을 수치화한다.\n\n')
lines.append('**안전괴리지수(SDI, Safety Decoupling Index)** = Substantive score − Formal score\n\n')
lines.append('| SDI 값 | 해석 |\n|---|---|\n')
lines.append('| SDI > 0 | 실질이 형식보다 강함 → **진짜 안전** |\n')
lines.append('| SDI ≈ 0 | 형식·실질 균형 |\n')
lines.append('| SDI < 0 | 형식만 있고 실질 부족 → **안전 형식주의(Safety Decoupling)** |\n\n')

# 2. 데이터 및 방법론
lines.append('## 2. 데이터 및 분석 방법\n\n')
lines.append('### 2-1. 데이터\n\n')
lines.append('| 업종 | 원본 파일 | 표본 수 | 코드북 시트 |\n|---|---|---|---|\n')
lines.append('| 건설업 | raw_data_건설업_230824.CSV | 1,502 | 건설업 |\n')
lines.append('| 제조업 | raw_data_제조업_230824.csv | 3,255 | 제조서비스업 |\n')
lines.append('| 서비스업 | raw_data_서비스업_230824.csv | 2,551 | 제조서비스업 |\n\n')

lines.append('### 2-2. 변수 정규화 및 방향 정렬\n\n')
lines.append('- **이진형 Formal 변수** (Q6, Q9, Q12 등): 1=예→1.0, 2=아니요→0.0\n')
lines.append('- **건설업 Q10** (위원회 운영): 위원회=1.0, 노사협의체=0.5, 미운영=0.0, 모름=NaN\n')
lines.append('- **제조/서비스 Q10**: 운영=1.0, 미운영=0.0, 잘모름=NaN\n')
lines.append('- **인원수 변수** (Q8_*): Min-Max 정규화\n')
lines.append('- **Likert 5점 척도** (Substantive 변수): 9→NaN 후 Min-Max 정규화\n\n')

lines.append('### 2-3. SDI 계산식\n\n')
lines.append('```\n')
lines.append('F-composite_k = 그룹 k 변수들의 행 평균 (각 변수 0~1 정규화)\n')
lines.append('S-composite_k = 그룹 k 변수들의 행 평균\n')
lines.append('F_score = mean(F-composite_1 ~ F-composite_7)   # 7그룹 평균\n')
lines.append('S_score = mean(S-composite_1 ~ S-composite_7)   # 7그룹 평균\n')
lines.append('SDI     = S_score - F_score\n')
lines.append('```\n\n')

lines.append('### 2-4. 음이항 GLM(GLM-NegBin) 회귀\n\n')
lines.append('업무상 사고 건수는 **과잉 0(excess zeros)**을 포함하는 카운트 자료이므로 일반 포아송 대신 음이항 GLM(GLM-NegBin)을 적용한다.\n\n')
lines.append('| 업종 | 사고 0건 비율 | 모형 선택 근거 |\n|---|---|---|\n')
lines.append('| 건설업 | 64.8% | 구조적 0(노출 없는 현장) + 표본 0 혼재 |\n')
lines.append('| 제조·서비스업 | 80.4% | 소규모 사무직 등 구조적 무사고 집단 비율 높음 |\n\n')
lines.append('**음이항 GLM 모형 구조 (log link, offset=log(종사자수)):**\n\n')
lines.append('```\n')
lines.append('log(E[Y]) = Xβ + log(종사자수)   [log link, offset=log_workers]\n')
lines.append('Var(Y)   = μ + α·μ²               [음이항 분산: Poisson보다 overdispersion 허용]\n\n')


lines.append('```\n\n')
lines.append('- **설명변수**: SDI + 제도적 통제변수(formal_inst 등) + 근로자 구성 비율 + 범주형 통제변수\n')

lines.append('- **추정 방법**: BFGS 최적화, 최대 500회 반복\n\n')

# 3. 변수 분류표
lines.append('## 3. 변수 분류 결과\n\n')
lines.append('### 3-1. 건설업\n\n')
lines.append('| 그룹ID | 원본변수코드 | 범주 | 변수수 |\n|---|---|---|---|\n')
for name, cols in {**F_CON_GROUPS, **S_CON_GROUPS}.items():
    cat = 'Formal(형식)' if name.startswith('F') else 'Substantive(실질)'
    lines.append(f'| {name} | `{" ".join(cols)}` | {cat} | {len(cols)} |\n')
lines.append('\n')
lines.append('### 3-2. 제조·서비스업\n\n')
lines.append('| 그룹ID | 원본변수코드 | 범주 | 변수수 |\n|---|---|---|---|\n')
for name, cols in {**F_MS_GROUPS, **S_MS_GROUPS}.items():
    cat = 'Formal(형식)' if name.startswith('F') else 'Substantive(실질)'
    lines.append(f'| {name} | `{" ".join(cols)}` | {cat} | {len(cols)} |\n')
lines.append('\n')

# 4. EFA 결과
lines.append('## 4. 탐색적 요인분석(EFA) 결과\n\n')
lines.append('| 지표 | 건설업 | 제조업 |\n|---|---|---|\n')
kmo_c = f'{kmo_con_val:.3f}' if kmo_con_val else 'N/A'
kmo_m = f'{kmo_ms_val:.3f}' if kmo_ms_val else 'N/A'
ev_c  = str(ev_con_top) if ev_con_top else 'N/A'
ev_m  = str(ev_ms_top) if ev_ms_top else 'N/A'
lines.append(f'| KMO | {kmo_c} | {kmo_m} |\n')
lines.append(f'| 처음 5개 고유값 | {ev_c} | {ev_m} |\n')
lines.append(f'| 투입 변수 수 | {len(S_CON_ALL)} | {len(S_MS_ALL)} |\n\n')
if loads_con is not None:
    lines.append('**건설업 EFA 적재량 (절대값 ≥ 0.4)**\n\n')
    lines.append('| 원본변수코드 | Factor1 | Factor2 | 주요요인 |\n|---|---|---|---|\n')
    for idx, row in loads_con[loads_con['최대적재']>=0.4].head(15).iterrows():
        lines.append(f'| {idx} | {row["Factor1"]:.3f} | {row["Factor2"]:.3f} | {row["주요요인"]} |\n')
    lines.append('\n')
if loads_ms is not None:
    lines.append('**제조업 EFA 적재량 (절대값 ≥ 0.4)**\n\n')
    lines.append('| 원본변수코드 | Factor1 | Factor2 | 주요요인 |\n|---|---|---|---|\n')
    for idx, row in loads_ms[loads_ms['최대적재']>=0.4].head(15).iterrows():
        lines.append(f'| {idx} | {row["Factor1"]:.3f} | {row["Factor2"]:.3f} | {row["주요요인"]} |\n')
    lines.append('\n')

# 5. SDI 기술통계
lines.append('## 5. SDI 기술통계\n\n')
lines.append('| 업종 | N | 평균(비가중) | 중앙값 | 표준편차 | 가중평균(WT2) | 방향 |\n|---|---|---|---|---|---|---|\n')
for ind, d in desc_stats.iterrows():
    wm = wm_vals.get(ind, 'N/A')
    direction = sdi_con_dir if ind=='건설' else (sdi_mfg_dir if ind=='제조' else sdi_svc_dir)
    lines.append(f'| {ind} | {int(d["N"])} | {d["평균"]:.4f} | {d["중앙값"]:.4f} | {d["표준편차"]:.4f} | {wm} | {direction} |\n')
lines.append('\n')
lines.append(f'**Kruskal-Wallis**: H={kw_stat:.3f}, p={"<0.001" if kw_p<0.001 else f"{kw_p:.4f}"} ')
lines.append('— 업종 간 SDI 분포 차이 ' + ('**통계적으로 유의**' if kw_p<0.05 else '유의하지 않음') + '\n\n')
lines.append('**형식주의 사업장 비율 (SDI < 0)**\n\n')
lines.append('| 업종 | SDI<0 사업장 수 | 비율 |\n|---|---|---|\n')
for ind, df_ in [('건설', df_con), ('제조', df_mfg), ('서비스', df_svc)]:
    neg = (df_['SDI'] < 0).sum(); tot = df_['SDI'].notna().sum()
    lines.append(f'| {ind} | {neg} | {neg/tot*100:.1f}% |\n')
lines.append('\n')

# 6. GLM-NegBin 회귀분석 결과
lines.append('## 6. 음이항 GLM(GLM-NegBin) 회귀분석 결과\n\n')
lines.append('**종속변수**: 업무상 사고 발생 건수 합산 (Q27_1_1~3 + Q27_3_1~3)  \n')
lines.append('**offset**: log(종사자 수) — 사고 노출량 통제  \n')
lines.append('**IRR 해석**: exp(β), 1 미만이면 사고율 감소 / 1 초과면 증가  \n')
lines.append('**음이항 GLM 회귀계수 해석**: IRR=exp(β), offset=log(종사자수) 적용\n\n')

def format_nb_table(summ, res, title, _unused=None):
    result = []
    result.append(f'### {title}\n\n')
    if summ is None:
        result.append('⚠ GLM-NegBin 회귀분석 결과 없음\n\n')
        return result
    result.append('**GLM-NegBin 회귀 결과**\n\n')
    result.append('| 변수 | 계수(β) | IRR | 95%CI(하) | 95%CI(상) | p값 | 유의성 |\n')
    result.append('|---|---|---|---|---|---|---|\n')
    for _, row in summ.iterrows():
        if row['변수'] == 'const': continue
        result.append(f'| {row["변수"]} | {row["계수(β)"]:.4f} | {row["IRR"]:.4f} | '
                      f'{row["CI_95%_lower"]:.4f} | {row["CI_95%_upper"]:.4f} | '
                      f'{row["p값"]:.4f} | {row["유의성"]} |\n')
    result.append('\n')
    sdi_r = summ[summ['변수']=='SDI']
    if not sdi_r.empty:
        r = sdi_r.iloc[0]
        direction = '사고율 감소 (SDI↑→사고↓, 가설 지지)' if r['계수(β)'] < 0 else '사고율 증가 (SDI↑→사고↑, 가설 불지지)'
        pct = abs((r['IRR']-1)*100)
        result.append(f'> **SDI**: β={r["계수(β)"]:.4f}, IRR={r["IRR"]:.4f}, p={r["p값"]:.4f} {r["유의성"]}  \n')
        result.append(f'> {direction}  \n')
        result.append(f'> SDI 1단위 증가 시 사고율 {"감소" if r["계수(β)"]<0 else "증가"} {pct:.1f}%\n\n')
    if res is not None:
        result.append(f'> **모형 적합도**: AIC={res.aic:.2f}, Log-L={res.llf:.2f}\n\n')
    return result

infl_con = ['const'] + [f'log_workers_{i}' if i>0 else 'const' for i in range(1)]
infl_con_actual = ['const', 'log_workers'] if res_con is not None else ['const']
infl_ms_actual  = ['const', 'log_workers', 'union'] if res_ms is not None else ['const']

lines += format_nb_table(summ_con, res_con, '6-1. 건설업 GLM-NegBin 회귀결과', infl_con_actual)
lines += format_nb_table(summ_ms,  res_ms,  '6-2. 제조·서비스업 GLM-NegBin 회귀결과', infl_ms_actual)

# 7. 결론
lines.append('## 7. 결론 및 시사점\n\n')
lines.append('### 7-1. 핵심 발견\n\n')
for ind, df_ in [('건설', df_con), ('제조', df_mfg), ('서비스', df_svc)]:
    neg_pct = (df_['SDI'] < 0).mean() * 100
    sdi_mean = df_['SDI'].mean()
    lines.append(f'- **{ind}업**: SDI 평균 {sdi_mean:.4f}, 형식주의 사업장(SDI<0) {neg_pct:.1f}%\n')
lines.append('\n')
lines.append('### 7-2. 정책 시사점\n\n')
lines.append('1. **형식주의 사업장 집중 관리**: SDI < 0인 사업장은 인증·서류는 보유했으나 실질 안전활동 부족 — 현장 점검 강화 대상\n')
lines.append('2. **경영진 안전 문화 개선**: Substantive 요인 중 경영진 안전 강조(S-C3/S-M2) 개선이 SDI 개선에 효과적\n')
lines.append('3. **업종별 맞춤 접근**: 건설업 SDI 평균이 가장 낮음 — 현장 실질 안전 관리 강화 우선 필요\n\n')
lines.append('### 7-3. 분석 한계\n\n')
lines.append('- 모형 수렴 안정성을 위해 음이항 GLM 적용; 추가 민감도 분석 권장\n')
lines.append('- F/S 변수 수 불균형은 그룹 composite 방식으로 해소했으나 민감도 분석 권장\n')
lines.append('- 횡단 연구 설계로 인해 SDI와 사고율 간 인과관계 주장에는 한계 존재\n\n')
lines.append('---\n\n')
lines.append('*보고서 자동 생성 완료 (SDI_analysis.ipynb 실행 결과)*\n')

report_text = ''.join(lines)
with open(OUT + 'SDI_report.md', 'w', encoding='utf-8') as f:
    f.write(report_text)

print(f'마크다운 보고서 저장: {OUT}SDI_report.md')
print(f'보고서 길이: {len(report_text):,}자')
print('\n=== 모든 분석 완료 ===')
for fname in sorted(os.listdir(OUT)):
    fpath = OUT + fname
    print(f'  {fname} ({os.path.getsize(fpath):,} bytes)')

마크다운 보고서 저장: c:/Users/USER/Desktop/SAB_paper/output/SDI_report.md
보고서 길이: 7,777자

=== 모든 분석 완료 ===


  SDI_report.md (10,435 bytes)
  efa_loadings_con.csv (866 bytes)
  efa_loadings_ms.csv (863 bytes)
  regression_results_con.csv (976 bytes)
  regression_results_ms.csv (706 bytes)
  sdi_comparison.png (81,092 bytes)
  sdi_descriptive_stats.csv (257 bytes)
  sdi_scores_con.csv (75,764 bytes)
  sdi_scores_mfg.csv (172,938 bytes)
  sdi_scores_svc.csv (129,896 bytes)
  variable_classification_con.csv (366 bytes)
  variable_classification_ms.csv (366 bytes)
